In [ ]:
from pathlib import Path
from typing import Callable

from application.user_params import user_params
import panel as pn
import param
import sqlalchemy
import panel_material_ui as pmui
from template import MyMaterialTemplate
import yaml

current_dir = Path(__file__).parent if '__file__' in globals() else Path('.')


In [ ]:
if (current_dir / 'config.yml').exists():
    with open(current_dir / 'config.yml', 'r', encoding='utf-8') as file:
        config = yaml.safe_load(file)

APPLICATION_NAME = 'Panel - Óptico'


In [ ]:
class App(param.Parameterized):
    """App"""
    __version_index__ = 1  # increment to reset user_params

    def __init__(self, get_dwh_dburi: Callable | None = None, **params):
        super().__init__(**params)
        # Conexión a base de datos (opcional). Es perezosa: create_engine no
        # conecta hasta que se usa. Se crea solo si se pasa get_dwh_dburi.
        self.engine = (
            sqlalchemy.create_engine(get_dwh_dburi(), echo=False)
            if get_dwh_dburi is not None
            else None
        )

    def onload(self) -> None:
        if pn.state.curdoc is not None:
            # we are in a panel server, not jupyter
            user_params.sync(self, [])
        # other processing on load

    def settings_panel(self) -> pn.reactive.Reactive:
        return pn.Param(
            self,
            parameters=[
            ],
            widgets={
                # Ejemplo: mapear parámetros a widgets de panel_material_ui
                # 'mi_param': pmui.Select,
                # 'otro_param': pmui.TextInput,
            }
        )

    def view(self) -> pn.reactive.Reactive:
        return pmui.Column()


In [ ]:
# Base de datos (opcional): cuando el informe necesite datos, descomenta el
# bloque y usa una sección que exista en config/.secrets.yaml (p.ej. 'navision').
# El helper depende del motor: mssql_url para SQL Server, postgres_url para
# PostgreSQL (application/config/urls.py).
#
# from application.config.config import Config
# from application.config.urls import mssql_url
#
# def get_dwh_dburi() -> str:
#     secrets = Config()['navision']
#     return mssql_url(secrets, APPLICATION_NAME)
#
# app = App(name=APPLICATION_NAME, get_dwh_dburi=get_dwh_dburi)

app = App(name=APPLICATION_NAME)

pn.state.onload(app.onload)

pn.config.js_files['flatpickr_i18n'] = 'https://npmcdn.com/flatpickr/dist/l10n/es.js'

if pn.state.curdoc is not None:
    # We are in a panel server
    pn.state.location.sync(app, [
    ])


In [ ]:
template = MyMaterialTemplate(title=app.name)
template.append_sidebar(pmui.Divider())
template.append_sidebar(app.settings_panel)
template.main.append(app.view)
if pn.state.curdoc is None:
    # Estamos en Jupyter: servimos el informe accesible por red.
    # address='0.0.0.0' escucha en todas las interfaces (por defecto es solo
    # localhost, inaccesible desde fuera) y websocket_origin='*' evita el 403
    # de origen al entrar por la IP/hostname. El puerto se imprime en la consola.
    template.show(address='0.0.0.0', port=0, websocket_origin='*', open=False)
else:
    template.servable()
